In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("--- Step 1: Loading Datasets ---")

# Defining Paths
data_dir = "../data/raw/"
ton_path = os.path.join(data_dir, "ton_iot", "Network_dataset_1.csv")
bot_path = os.path.join(data_dir, "bot_iot", "bot_iot_mapped.csv")
cic_path = os.path.join(data_dir, "cic_ids2017", "cic_ids2017_mapped.csv")

# We'll sample 50,000 rows from each to keep memory manageable and balance the classes
SAMPLE_SIZE = 50000

print("Loading ToN-IoT...")
df_ton = pd.read_csv(ton_path)
# ToN-IoT uses 'label' (0 or 1)
df_ton = df_ton.sample(n=min(SAMPLE_SIZE, len(df_ton)), random_state=42)

print("Loading mapped BoT-IoT...")
df_bot = pd.read_csv(bot_path)
df_bot = df_bot.sample(n=min(SAMPLE_SIZE, len(df_bot)), random_state=42)

print("Loading mapped CIC-IDS2017...")
df_cic = pd.read_csv(cic_path)
df_cic = df_cic.sample(n=min(SAMPLE_SIZE, len(df_cic)), random_state=42)

print(f"Data loaded! ToN: {len(df_ton)}, BoT: {len(df_bot)}, CIC: {len(df_cic)}")

--- Step 1: Loading Datasets ---
Loading ToN-IoT...
Loading mapped BoT-IoT...
Loading mapped CIC-IDS2017...
Data loaded! ToN: 50000, BoT: 50000, CIC: 50000


In [3]:
print("--- Step 2: Standardizing Columns and Concatenating ---")

# We need to make sure all dataframes use the same features before we merge.
expected_features = [
    'duration', 'src_bytes', 'dst_bytes', 'missed_bytes', 'src_pkts',
    'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype',
    'dns_rcode', 'http_request_body_len', 'http_response_body_len',
    'http_status_code'
]

def prep_dataframe(df, dataset_name):
    # Fill missing expected numeric columns with 0
    for col in expected_features:
        if col not in df.columns:
            df[col] = 0.0
            
    # Get label. ToN-IoT uses 'label', CIC/BoT might use 'Label'
    label_col = 'Label' if 'Label' in df.columns else 'label'
    
    # Convert to binary (0 = Normal, 1 = Attack)
    y = df[label_col].astype(str).apply(lambda x: 0 if str(x).lower() in ['0', '0.0', 'normal', 'benign', 'nan'] else 1)
    
    # Extract numeric X
    X = df[expected_features].copy()
    
    # Force convert to numeric, turning strings like '0.0.0.0' or '-' into NaN
    for col in expected_features:
        X[col] = pd.to_numeric(X[col], errors='coerce')
        
    X.fillna(0, inplace=True)
    
    # Note: For one-hot encoded categorical columns (proto, conn_state), we'd need to align them perfectly. 
    # For simplicity in this Omni Model, we'll stick to the core numerical identifiers.
    
    return X, y

X_ton, y_ton = prep_dataframe(df_ton, 'ToN-IoT')
X_bot, y_bot = prep_dataframe(df_bot, 'BoT-IoT')
X_cic, y_cic = prep_dataframe(df_cic, 'CIC-IDS2017')

# Stack them!
X_omni = pd.concat([X_ton, X_bot, X_cic], ignore_index=True)
y_omni = pd.concat([y_ton, y_bot, y_cic], ignore_index=True)

print(f"Unified Omni Dataset Shape: {X_omni.shape}")
print("Class Balance:\n", y_omni.value_counts(normalize=True))

--- Step 2: Standardizing Columns and Concatenating ---
Unified Omni Dataset Shape: (150000, 14)
Class Balance:
 label
1    0.661947
0    0.338053
Name: proportion, dtype: float64


In [4]:
print("--- Step 3: Training the Omni Model ---")

X_train, X_test, y_train, y_test = train_test_split(X_omni, y_omni, test_size=0.2, random_state=42, stratify=y_omni)

rf_omni = RandomForestClassifier(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42)

print("Training in progress (this may take a minute)...")
rf_omni.fit(X_train, y_train)
print("Training Complete!")

--- Step 3: Training the Omni Model ---
Training in progress (this may take a minute)...
Training Complete!


In [5]:
print("--- Step 4: Evaluation ---")
y_pred = rf_omni.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f"Omni Model Accuracy: {accuracy:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Format the model output for the SENTRi-X expected dimensions in real-time inference
# We'll pad out the categorical columns expected by main.py with dummy zeros temporarily
# so the backend shape matches perfectly without crashing.
dummy_features = [
    'proto_tcp', 'proto_udp', 'conn_state_REJ', 
    'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR',      
    'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2',
    'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR'
]

X_omni_full = pd.concat([X_omni, pd.DataFrame(0, index=X_omni.index, columns=dummy_features)], axis=1)

print("\nRetraining with padded feature dimensions to match backend expectations...")
rf_omni_deploy = RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=-1, random_state=42)
rf_omni_deploy.fit(X_omni_full, y_omni)

# Export to Models folder
model_path = os.path.join("..", "models", "rf_model_omni.joblib")
joblib.dump(rf_omni_deploy, model_path)

print(f"\nSUCCESS! Omni Model saved to {model_path}")

--- Step 4: Evaluation ---
Omni Model Accuracy: 99.41%

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.99      0.99     10142
           1       0.99      1.00      1.00     19858

    accuracy                           0.99     30000
   macro avg       0.99      0.99      0.99     30000
weighted avg       0.99      0.99      0.99     30000


Retraining with padded feature dimensions to match backend expectations...

SUCCESS! Omni Model saved to ..\models\rf_model_omni.joblib


In [6]:
print("--- Step 5: Training the Omni CNN Model ---")
# Import TensorFlow/Keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# The CNN requires a 3D input shape (samples, features, channels)
# We will use X_omni_full to ensure it matches the 28 padded features exactly.
X_cnn = X_omni_full.values.reshape((X_omni_full.shape[0], X_omni_full.shape[1], 1))

print(f"CNN Input Shape for training: {X_cnn.shape}")

# Build the 1D CNN Architecture
cnn_omni = Sequential([
    Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_cnn.shape[1], 1)),
    MaxPooling1D(pool_size=2),
    Dropout(0.2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Binary classification output
])

# Compile the model
cnn_omni.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Define Early Stopping Callback
# Restores the mathematically best weights based on validation loss, not training loss
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,                 # Wait for 3 epochs before giving up
    restore_best_weights=True,  # Crucial for returning optimal model
    verbose=1
)

# Train the CNN with an extended potential max epochs of 20
print("\nTraining CNN Omni Model (Max 20 Epochs with Early Stopping)...")
cnn_omni.fit(
    X_cnn, 
    y_omni, 
    epochs=20, 
    batch_size=64, 
    validation_split=0.2, 
    callbacks=[early_stopping]
)

# Export the CNN to the models directory
cnn_model_path = os.path.join("..", "models", "cnn_model_omni.h5")
cnn_omni.save(cnn_model_path)

print(f"\nSUCCESS! Omni CNN Model saved to {cnn_model_path}")

--- Step 5: Training the Omni CNN Model ---
CNN Input Shape for training: (150000, 28, 1)

Training CNN Omni Model (Max 20 Epochs with Early Stopping)...
Epoch 1/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 7ms/step - accuracy: 0.9258 - loss: 3247.1125 - val_accuracy: 0.8029 - val_loss: 0.6973
Epoch 2/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9490 - loss: 22.5324 - val_accuracy: 0.7981 - val_loss: 0.5043
Epoch 3/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9513 - loss: 3.2571 - val_accuracy: 0.7980 - val_loss: 0.5098
Epoch 4/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9529 - loss: 2.1754 - val_accuracy: 0.8304 - val_loss: 0.4490
Epoch 5/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step - accuracy: 0.9550 - loss: 1.1386 - val_accuracy: 0.8342 - val_loss: 0.4283
Epoch 6/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 6ms/step - accuracy: 0.9578 - loss: 0.6345 - val_accuracy: 0.8372 - val_loss: 0.4169
Epoch 7/20
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 12s 7ms/step -


SUCCESS! Omni CNN Model saved to ..\models\cnn_model_omni.h5


In [9]:
print("--- Step 6: Final CNN Evaluation (Omni Dataset) ---")
X_test_padded = X_omni_full.loc[X_test.index]
X_test_cnn = X_test_padded.values.reshape((X_test_padded.shape[0], X_test_padded.shape[1], 1))

from sklearn.metrics import classification_report, accuracy_score

start_inference = pd.Timestamp.now()
cnn_probs = cnn_omni.predict(X_test_cnn).ravel()
cnn_preds = (cnn_probs >= 0.5).astype(int)
end_inference = pd.Timestamp.now()

cnn_acc = accuracy_score(y_test, cnn_preds) * 100
print(f"Omni CNN Model Accuracy: {cnn_acc:.4f}%")
print("\nCNN Classification Report:\n", classification_report(y_test, cnn_preds, digits=4))

--- Step 6: Final CNN Evaluation (Omni Dataset) ---
938/938 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
Omni CNN Model Accuracy: 95.3600%

CNN Classification Report:
               precision    recall  f1-score   support

           0     0.8844    0.9924    0.9353     10142
           1     0.9959    0.9338    0.9638     19858

    accuracy                         0.9536     30000
   macro avg     0.9402    0.9631    0.9496     30000
weighted avg     0.9582    0.9536    0.9542     30000



In [10]:
print("\n--- Step 7: SENTRi-X Hybrid Engine Performance (Omni Dataset) ---")
# Get RF probabilities from the deployed model which uses the padded features
rf_probs = rf_omni_deploy.predict_proba(X_test_padded)[:, 1]

# Soft-Voting Ensemble
ensemble_probs = (rf_probs + cnn_probs) / 2.0
ensemble_preds = (ensemble_probs >= 0.5).astype(int)

ensemble_acc = accuracy_score(y_test, ensemble_preds) * 100
print(f"Omni Hybrid Ensemble Accuracy: {ensemble_acc:.4f}%")
print("\nHybrid Ensemble Classification Report:\n", classification_report(y_test, ensemble_preds, digits=4))


--- Step 7: SENTRi-X Hybrid Engine Performance (Omni Dataset) ---
Omni Hybrid Ensemble Accuracy: 99.3600%

Hybrid Ensemble Classification Report:
               precision    recall  f1-score   support

           0     0.9903    0.9907    0.9905     10142
           1     0.9953    0.9951    0.9952     19858

    accuracy                         0.9936     30000
   macro avg     0.9928    0.9929    0.9929     30000
weighted avg     0.9936    0.9936    0.9936     30000

